In [2]:
import sys

print("Python version:", sys.version)
print("RepoSage environment is ready!")
%pip install pandas scikit-learn requests

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
RepoSage environment is ready!


In [3]:
import requests

url = "https://huggingface.co/api/datasets/amazeng-co/git-jit-defects"

response = requests.get(url)

print("Status code:", response.status_code)
print("Dataset accessible:", response.status_code == 200)


Status code: 200
Dataset accessible: True


In [4]:
import pandas as pd

url = "https://huggingface.co/datasets/amazeng-co/git-jit-defects/resolve/main/commits.parquet"

df = pd.read_parquet(url)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (360463, 18)

Columns:
['repo', 'sha', 't', 'author', 'subj', 'nfiles', 'la', 'ld', 'hunks', 'ndirs', 'nexts', 'test_ratio', 'a_prior_n', 'a_prior_rate', 'a_days', 'msg_len', 'msg_ref', 'y']

First 5 rows:


,repo,sha,t,author,subj,nfiles,la,ld,hunks,ndirs,nexts,test_ratio,a_prior_n,a_prior_rate,a_days,msg_len,msg_ref,y
0,AdguardTeam/AdGuardHome,bed92f89f04ba19e03d88ad8218c67c5928b5530,1535979320,422d2dafdfea78bd,add client column and tooltip to blocked requests,4,49,8,14,3,2,0.0,1,0.400000,2.892141,49,0,1
1,AdguardTeam/AdGuardHome,fa8bc570827cae708266901754712fb0c9a1f1ea,1536160838,422d2dafdfea78bd,add reason status,1,13,1,2,1,1,0.0,2,0.500000,4.993044,17,0,1
2,AdguardTeam/AdGuardHome,606bed9d202eeb67eae8f9d38f1f0fc31220011d,1536171734,d7450458c1bcd694,Reduce binary size of coredns by 60% by removi...,1,1,0,1,1,1,0.0,4,0.250000,6.095266,123,0,0
3,AdguardTeam/AdGuardHome,e122d9138bd117d2f0e314dba4421a64adba5c71,1536171911,d7450458c1bcd694,"sometimes answer can be empty, therefore quest...",1,19,17,7,1,1,0.0,5,0.222222,6.097315,88,0,0
4,AdguardTeam/AdGuardHome,33fbccf0badee8a4c5f790d850d85e161536779f,1536188444,d7450458c1bcd694,Move starting of coredns server into separate ...,1,24,13,10,1,1,0.0,6,0.200000,6.288669,54,0,0


In [6]:
# Check the number of defective and non-defective commits
print("Label distribution:")
print(df["y"].value_counts())

print("\nLabel percentages:")
print(df["y"].value_counts(normalize=True) * 100)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate commits
print("\nDuplicate commit IDs:", df["sha"].duplicated().sum())

# Check the time range
df["date"] = pd.to_datetime(df["t"], unit="s", utc=True)

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())


Label distribution:
y
0    318340
1     42123
Name: count, dtype: int64

Label percentages:
y
0    88.314196
1    11.685804
Name: proportion, dtype: float64

Missing values:
repo            0
sha             0
t               0
author          0
subj            0
nfiles          0
la              0
ld              0
hunks           0
ndirs           0
nexts           0
test_ratio      0
a_prior_n       0
a_prior_rate    0
a_days          0
msg_len         0
msg_ref         0
y               0
date            0
dtype: int64

Duplicate commit IDs: 0

Date range:
2005-09-20 21:42:20+00:00 to 2026-04-18 18:38:19+00:00


In [7]:
# Check how many repositories are represented
print("Number of repositories:", df["repo"].nunique())

print("\nLargest repositories:")
print(df["repo"].value_counts().head(10))

# Sort commits chronologically
df = df.sort_values("t").reset_index(drop=True)

# Use the earliest 80% for training
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print("\nTraining period:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTesting period:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nTraining defect rate:")
print(round(train_df["y"].mean() * 100, 2), "%")

print("\nTesting defect rate:")
print(round(test_df["y"].mean() * 100, 2), "%")


Number of repositories: 48

Largest repositories:
repo
SerenityOS/serenity               63698
influxdata/influxdb               35680
fish-shell/fish-shell             19948
facebook/folly                    14664
ankitects/anki                    14281
jj-vcs/jj                         13996
Lightning-AI/pytorch-lightning    12281
uutils/coreutils                  11937
ValveSoftware/Proton              10475
huggingface/diffusers             10030
Name: count, dtype: int64

Training period:
2005-09-20 21:42:20+00:00 to 2024-11-23 10:18:49+00:00

Testing period:
2024-11-23 10:24:03+00:00 to 2026-04-18 18:38:19+00:00

Training defect rate:
11.55 %

Testing defect rate:
12.21 %


In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# Select the features for our first model
features = [
    "nfiles",
    "la",
    "ld",
    "hunks",
    "ndirs",
    "nexts",
    "test_ratio",
    "msg_len",
    "msg_ref"
]

# Prepare training and testing data
X_train = train_df[features]
y_train = train_df["y"]

X_test = test_df[features]
y_test = test_df["y"]

# Create the baseline model
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

# Train the model
model.fit(X_train, y_train)

# Predict defect probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Predict defect labels
y_pred = (y_prob >= 0.5).astype(int)

# Evaluate the model
print("BASELINE MODEL RESULTS")
print("-----------------------")
print("PR-AUC:", round(average_precision_score(y_test, y_prob), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

BASELINE MODEL RESULTS
-----------------------
PR-AUC: 0.2712
Precision: 0.2022
Recall: 0.682

Confusion Matrix:
[[39587 23700]
 [ 2800  6006]]


In [9]:
print("df exists:", "df" in globals())
print("train_df exists:", "train_df" in globals())
print("test_df exists:", "test_df" in globals())
print("model exists:", "model" in globals())

df exists: True
train_df exists: True
test_df exists: True
model exists: True


In [10]:
print("Full dataset:", df.shape)
print("Training set:", train_df.shape)
print("Test set:", test_df.shape)
print("Model:", type(model).__name__)

Full dataset: (360463, 19)
Training set: (288370, 19)
Test set: (72093, 19)
Model: Pipeline


In [11]:
print("Training dates:", train_df["date"].min(), "to", train_df["date"].max())
print("Test dates:    ", test_df["date"].min(), "to", test_df["date"].max())

print("\nTraining defect rate:")
print(train_df["y"].mean())

print("\nTest defect rate:")
print(test_df["y"].mean())

Training dates: 2005-09-20 21:42:20+00:00 to 2024-11-23 10:18:49+00:00
Test dates:     2024-11-23 10:24:03+00:00 to 2026-04-18 18:38:19+00:00

Training defect rate:
0.11553559662933037

Test defect rate:
0.12214778133799398


In [12]:
print("Features used:")
print(features)

print("\nAre label or commit ID included?")
print("y in features:", "y" in features)
print("sha in features:", "sha" in features)

Features used:
['nfiles', 'la', 'ld', 'hunks', 'ndirs', 'nexts', 'test_ratio', 'msg_len', 'msg_ref']

Are label or commit ID included?
y in features: False
sha in features: False


In [13]:
import numpy as np

print("Missing values in selected features:")
print(train_df[features].isna().sum())

print("\nInfinite values in selected features:")
print(np.isinf(train_df[features].select_dtypes(include="number")).sum())

Missing values in selected features:
nfiles        0
la            0
ld            0
hunks         0
ndirs         0
nexts         0
test_ratio    0
msg_len       0
msg_ref       0
dtype: int64

Infinite values in selected features:
nfiles        0
la            0
ld            0
hunks         0
ndirs         0
nexts         0
test_ratio    0
msg_len       0
msg_ref       0
dtype: int64


In [22]:
# Keep the existing final test set untouched.
# Split each repository's earlier training data chronologically.

train_parts = []
val_parts = []

for repo, repo_df in train_df.groupby("repo", sort=False):
    repo_df = repo_df.sort_values("t")
    split_index = int(len(repo_df) * 0.8)

    train_parts.append(repo_df.iloc[:split_index])
    val_parts.append(repo_df.iloc[split_index:])

new_train_df = pd.concat(train_parts).sort_values("t").copy()
val_df = pd.concat(val_parts).sort_values("t").copy()

print("New training set:", new_train_df.shape)
print("Validation set:", val_df.shape)
print("Final test set:", test_df.shape)

New training set: (230679, 19)
Validation set: (57691, 19)
Final test set: (72093, 19)


In [23]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score

X_train = new_train_df[features]
y_train = new_train_df["y"]

X_val = val_df[features]
y_val = val_df["y"]

baseline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

baseline_model.fit(X_train, y_train)

val_probs = baseline_model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print("Validation PR-AUC:", round(average_precision_score(y_val, val_probs), 4))
print("Validation precision:", round(precision_score(y_val, val_preds), 4))
print("Validation recall:", round(recall_score(y_val, val_preds), 4))

Validation PR-AUC: 0.1861
Validation precision: 0.1355
Validation recall: 0.6394


In [24]:
print("Validation defect rate:", round(y_val.mean(), 4))
print("Validation rows:", len(y_val))

Validation defect rate: 0.0874
Validation rows: 57691


In [25]:
from sklearn.metrics import precision_score, recall_score

for threshold in [0.3, 0.5, 0.7, 0.9]:
    val_preds_at_threshold = (val_probs >= threshold).astype(int)

    print(
        f"Threshold {threshold}: "
        f"precision={precision_score(y_val, val_preds_at_threshold, zero_division=0):.3f}, "
        f"recall={recall_score(y_val, val_preds_at_threshold, zero_division=0):.3f}, "
        f"flagged={val_preds_at_threshold.sum()}"
    )

Threshold 0.3: precision=0.090, recall=1.000, flagged=55782
Threshold 0.5: precision=0.135, recall=0.639, flagged=23800
Threshold 0.7: precision=0.257, recall=0.256, flagged=5033
Threshold 0.9: precision=0.300, recall=0.065, flagged=1085


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score

X_train = new_train_df[features]
y_train = new_train_df["y"]

X_val = val_df[features]
y_val = val_df["y"]

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_val_probs = rf_model.predict_proba(X_val)[:, 1]
rf_val_preds = (rf_val_probs >= 0.5).astype(int)

print("Random Forest validation PR-AUC:",
      round(average_precision_score(y_val, rf_val_probs), 4))
print("Random Forest validation precision:",
      round(precision_score(y_val, rf_val_preds, zero_division=0), 4))
print("Random Forest validation recall:",
      round(recall_score(y_val, rf_val_preds, zero_division=0), 4))

Random Forest validation PR-AUC: 0.3263
Random Forest validation precision: 0.2067
Random Forest validation recall: 0.7533


In [27]:
from sklearn.metrics import precision_score, recall_score

for name, probs in [
    ("Logistic Regression", val_probs),
    ("Random Forest", rf_val_probs),
]:
    print(f"\n{name}")
    for threshold in [0.3, 0.5, 0.7, 0.9]:
        preds = (probs >= threshold).astype(int)

        print(
            f"Threshold {threshold}: "
            f"precision={precision_score(y_val, preds, zero_division=0):.3f}, "
            f"recall={recall_score(y_val, preds, zero_division=0):.3f}, "
            f"flagged={preds.sum()}"
        )


Logistic Regression
Threshold 0.3: precision=0.090, recall=1.000, flagged=55782
Threshold 0.5: precision=0.135, recall=0.639, flagged=23800
Threshold 0.7: precision=0.257, recall=0.256, flagged=5033
Threshold 0.9: precision=0.300, recall=0.065, flagged=1085

Random Forest
Threshold 0.3: precision=0.144, recall=0.915, flagged=32112
Threshold 0.5: precision=0.207, recall=0.753, flagged=18372
Threshold 0.7: precision=0.305, recall=0.488, flagged=8075
Threshold 0.9: precision=0.599, recall=0.078, flagged=656


In [28]:
print(train_df.groupby("repo")["t"].agg(["min", "max"]).head())
print(val_df.groupby("repo")["t"].agg(["min", "max"]).head())

                                              min         max
repo                                                         
AdguardTeam/AdGuardHome                1535979320  1732285127
Lightning-AI/pytorch-lightning         1554304731  1732146464
NginxProxyManager/nginx-proxy-manager  1518501104  1732267858
ReactiveX/rxjs                         1428293197  1719557138
SagerNet/sing-box                      1656934798  1732335406
                                              min         max
repo                                                         
AdguardTeam/AdGuardHome                1664470293  1732285127
Lightning-AI/pytorch-lightning         1681135330  1732146464
NginxProxyManager/nginx-proxy-manager  1690251147  1732267858
ReactiveX/rxjs                         1601420602  1719557138
SagerNet/sing-box                      1729603702  1732335406


In [21]:
train_max = train_df.groupby("repo")["t"].max()
val_min = val_df.groupby("repo")["t"].min()

overlap = train_max[train_max >= val_min]

print("Repositories with train/validation time overlap:", len(overlap))
print(overlap)

Repositories with train/validation time overlap: 42
repo
AdguardTeam/AdGuardHome                  1732285127
Lightning-AI/pytorch-lightning           1732146464
NginxProxyManager/nginx-proxy-manager    1732267858
ReactiveX/rxjs                           1719557138
SagerNet/sing-box                        1732335406
SerenityOS/serenity                      1732330932
ValveSoftware/Proton                     1732329290
ankitects/anki                           1731994301
catchorg/Catch2                          1732027646
clockworklabs/SpacetimeDB                1732310009
date-fns/date-fns                        1730360186
dgtlmoon/changedetection.io              1732175906
dmlc/xgboost                             1732259243
dragonflydb/dragonfly                    1732268410
drizzle-team/drizzle-orm                 1732273325
facebook/folly                           1732308312
firecracker-microvm/firecracker          1732283493
fish-shell/fish-shell                    1732315724
floatin

In [29]:
train_max = new_train_df.groupby("repo")["t"].max()
val_min = val_df.groupby("repo")["t"].min()

overlap = train_max[train_max >= val_min]
print("Repositories with train/validation time overlap:", len(overlap))
print(overlap)

Repositories with train/validation time overlap: 2
repo
SagerNet/sing-box            1729603702
clockworklabs/SpacetimeDB    1722971177
Name: t, dtype: int64


In [30]:
for repo in ["SagerNet/sing-box", "clockworklabs/SpacetimeDB"]:
    tr = new_train_df[new_train_df["repo"] == repo]
    va = val_df[val_df["repo"] == repo]

    print(repo)
    print("Train max:", tr["t"].max())
    print("Validation min:", va["t"].min())
    print("Train rows at/after validation min:",
          (tr["t"] >= va["t"].min()).sum())
    print("Validation rows at/before train max:",
          (va["t"] <= tr["t"].max()).sum())

SagerNet/sing-box
Train max: 1729603702
Validation min: 1729603702
Train rows at/after validation min: 19
Validation rows at/before train max: 7
clockworklabs/SpacetimeDB
Train max: 1722971177
Validation min: 1722971177
Train rows at/after validation min: 2
Validation rows at/before train max: 2


In [31]:
train_parts = []
val_parts = []

for repo, repo_df in train_df.groupby("repo", sort=False):
    repo_df = repo_df.sort_values("t")
    split_index = int(len(repo_df) * 0.8)

    # Put the entire boundary timestamp in validation.
    cutoff = repo_df["t"].iloc[split_index]

    train_parts.append(repo_df[repo_df["t"] < cutoff])
    val_parts.append(repo_df[repo_df["t"] >= cutoff])

new_train_df = pd.concat(train_parts).sort_values("t").copy()
val_df = pd.concat(val_parts).sort_values("t").copy()

In [32]:
train_max = new_train_df.groupby("repo")["t"].max()
val_min = val_df.groupby("repo")["t"].min()

overlap = train_max[train_max >= val_min]
print(overlap)

Series([], Name: t, dtype: int64)


In [33]:
train_max = new_train_df.groupby("repo")["t"].max()
val_min = val_df.groupby("repo")["t"].min()

overlap = train_max[train_max >= val_min]
print(overlap)
print("Train shape:", new_train_df.shape)
print("Validation shape:", val_df.shape)

Series([], Name: t, dtype: int64)
Train shape: (230658, 19)
Validation shape: (57712, 19)


In [34]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score

# Use the corrected train/validation split
X_train = new_train_df[features]
y_train = new_train_df["y"]
X_val = val_df[features]
y_val = val_df["y"]

# Train Logistic Regression
lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)
lr_model.fit(X_train, y_train)
lr_probs = lr_model.predict_proba(X_val)[:, 1]

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_val)[:, 1]

# Print validation results
for name, probs in [
    ("Logistic Regression", lr_probs),
    ("Random Forest", rf_probs)
]:
    print(f"\n{name}")
    print("Validation PR-AUC:", round(average_precision_score(y_val, probs), 4))

    for threshold in [0.5, 0.7, 0.9]:
        preds = (probs >= threshold).astype(int)
        print(
            f"Threshold {threshold}: "
            f"precision={precision_score(y_val, preds, zero_division=0):.3f}, "
            f"recall={recall_score(y_val, preds, zero_division=0):.3f}, "
            f"flagged={preds.sum()}"
        )


Logistic Regression
Validation PR-AUC: 0.1861
Threshold 0.5: precision=0.136, recall=0.639, flagged=23813
Threshold 0.7: precision=0.256, recall=0.256, flagged=5038
Threshold 0.9: precision=0.301, recall=0.065, flagged=1084

Random Forest
Validation PR-AUC: 0.3257
Threshold 0.5: precision=0.208, recall=0.757, flagged=18349
Threshold 0.7: precision=0.301, recall=0.487, flagged=8153
Threshold 0.9: precision=0.588, recall=0.074, flagged=639


In [35]:
from sklearn.metrics import average_precision_score, precision_score, recall_score

X_test = test_df[features]
y_test = test_df["y"]

test_probs = rf_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.7).astype(int)

print("Final test PR-AUC:", round(average_precision_score(y_test, test_probs), 4))
print("Threshold: 0.7")
print("Precision:", round(precision_score(y_test, test_preds, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, test_preds, zero_division=0), 3))
print("Commits flagged:", int(test_preds.sum()))
print("Total test commits:", len(test_df))

Final test PR-AUC: 0.4129
Threshold: 0.7
Precision: 0.365
Recall: 0.593
Commits flagged: 14288
Total test commits: 72093
